# Data Generation Pipeline — Fine-tuning Vietnamese Bi-Encoder

Pipeline sinh dữ liệu QA từ PDF giáo trình để fine-tune `bkai-foundation-models/vietnamese-bi-encoder`.

| Tham số | Giá trị |
|---------|---------|
| Model sinh QA | `gemini-3.1-flash-lite` |
| Queries / chunk | 3 (factual + relational + applicative) |
| Target QA cuối | ~1,500 cặp sạch |
| Chunking | `underthesea.sent_tokenize` + sliding window |
| Negatives | easy / medium / hard (BM25) |
| Loss function (fine-tune) | `MultipleNegativesRankingLoss` |

```
PDF → Clean → Segment → Chunk → QA (Gemini) → Negatives → Filter → Split → Export
```

In [1]:
%pip install pymupdf underthesea rank-bm25 google-genai python-dotenv --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import re
import json
import time
import random
import hashlib
import logging
from pathlib import Path
from typing import List, Dict, Tuple
from collections import Counter

from dotenv import load_dotenv
load_dotenv()

# ── CONFIG (chỉnh tại đây) ────────────────────────────────────────────────────
PDF_PATH            = "../data/triet-hoc-mac-lenin-giao-trinh.pdf"
OUTPUT_DIR          = Path("../data/training_data_th")
CHECKPOINT_PATH     =  OUTPUT_DIR / ("qa_checkpoint.jsonl")

# Chunking
MAX_SENTENCES       = 7        # số câu tối đa mỗi chunk
OVERLAP_SENTENCES   = 1        # số câu chồng lấn giữa các chunk
MIN_CHUNK_CHARS     = 250      # loại chunk quá ngắn
MAX_CHUNK_CHARS     = 900      # cắt chunk quá dài
TARGET_CHUNKS       = 650      # mục tiêu số chunk
 
# QA Generation
QUERIES_PER_CHUNK   = 3        # 3 loại intent: factual / relational / applicative
GEMINI_MODEL        = "gemini-2.5-flash-lite"
GEMINI_TEMPERATURE  = 0.3
MAX_RETRIES         = 3
SLEEP_BETWEEN_CALLS = 0.6      # giây, tránh rate limit

# Negatives
NEGATIVES_PER_PAIR  = 3        # 1 easy + 1 medium + 1 hard

# Quality filter
MIN_QUERY_CHARS     = 20
MAX_QUERY_CHARS     = 280

# Split
TRAIN_RATIO         = 0.80
VAL_RATIO           = 0.10
# TEST = phần còn lại (~0.10)

RANDOM_SEED         = 42

# ─────────────────────────────────────────────────────────────────────────────
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger(__name__)

est_raw   = TARGET_CHUNKS * QUERIES_PER_CHUNK
est_clean = int(est_raw * 0.80)

print("✅ Config loaded")
print(f"   PDF            : {PDF_PATH}")
print(f"   Output dir     : {OUTPUT_DIR}")
print(f"   Target chunks  : {TARGET_CHUNKS}")
print(f"   Queries/chunk  : {QUERIES_PER_CHUNK}")
print(f"   Est. QA raw    : {est_raw}")
print(f"   Est. QA clean  : {est_clean}  (sau filter ~80%)")

✅ Config loaded
   PDF            : ../data/triet-hoc-mac-lenin-giao-trinh.pdf
   Output dir     : ..\data\training_data_th
   Target chunks  : 650
   Queries/chunk  : 3
   Est. QA raw    : 1950
   Est. QA clean  : 1560  (sau filter ~80%)


## Bước 1–2: Load PDF & Cleaning

- **Load**: đọc từng trang bằng `PyMuPDF` (text-based PDF).
- **Clean**: loại watermark, số trang, header/footer lặp, ghép dòng bị vỡ câu.

In [3]:
import fitz  # PyMuPDF


def load_pdf_pages(pdf_path: str) -> List[Dict]:
    """Trích text từng trang PDF, giữ metadata trang."""
    doc = fitz.open(pdf_path)
    pages = []
    for i in range(len(doc)):
        text = doc[i].get_text("text")
        pages.append({"page": i + 1, "text": text})
    doc.close()
    return pages


def clean_page_text(text: str) -> str:
    """
    Làm sạch artifact phổ biến của PDF giáo trình:
      - Watermark lOMoAR, studocu, downloaded by...
      - Số trang đứng độc lập
      - Dòng bị vỡ giữa câu (ghép lại)
      - Khoảng trắng thừa
    """
    # Watermark patterns
    text = re.sub(r"lOMoARcPSD\|[\w]+", "", text)
    text = re.sub(r"Downloaded by .+?(@.+?)?(\n|$)", "", text, flags=re.IGNORECASE)
    text = re.sub(r"Scan to open on .+?(\n|$)", "", text, flags=re.IGNORECASE)
    text = re.sub(r"Studocu is not .+?(\n|$)", "", text, flags=re.IGNORECASE)

    # Số trang / footer số đứng độc lập (dòng chỉ có số)
    text = re.sub(r"^\s*\d{1,4}\s*$", "", text, flags=re.MULTILINE)

    # Ghép dòng bị vỡ: dòng không kết thúc bằng dấu câu + dòng tiếp theo
    # không phải tiêu đề (không bắt đầu bằng CHỮ HOA liên tiếp hoặc số La Mã)
    text = re.sub(
        r"(?<![.!?:;,»\-–])\n(?![A-ZÀÁẢÃẠĂẮẰẶẲẴÂẤẦẨẪẬĐÊẾỀỂỄỆ\d\-–•])",
        " ",
        text,
    )

    # Khoảng trắng thừa
    text = re.sub(r"[ \t]{2,}", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


# ── Chạy ─────────────────────────────────────────────────────────────────────
pages = load_pdf_pages(PDF_PATH)
raw_text = "\n\n".join(p["text"] for p in pages)
clean_text = clean_page_text(raw_text)

removed_chars = len(raw_text) - len(clean_text)

print(f"✅ Load PDF xong")
print(f"   Số trang          : {len(pages)}")
print(f"   Raw text length   : {len(raw_text):,} ký tự")
print(f"   Clean text length : {len(clean_text):,} ký tự")
print(f"   Đã xóa            : {removed_chars:,} ký tự  ({removed_chars/len(raw_text)*100:.1f}%)")
print()

# Kiểm tra chất lượng: tỷ lệ dòng bất thường
lines = clean_text.splitlines()
very_short = [l for l in lines if 0 < len(l.strip()) < 10]
print(f"── Spot-check dòng quá ngắn (< 10 ký tự) ──────────")
print(f"   Tổng dòng          : {len(lines):,}")
print(f"   Dòng quá ngắn     : {len(very_short)} ({len(very_short)/len(lines)*100:.1f}%)")
if very_short[:5]:
    print(f"   Mẫu: {very_short[:5]}")
print()

# Sample 800 ký tự đầu
print("── Sample text (500 ký tự đầu) ─────────────────────")
print(clean_text[:500])

✅ Load PDF xong
   Số trang          : 214
   Raw text length   : 589,579 ký tự
   Clean text length : 576,320 ký tự
   Đã xóa            : 13,259 ký tự  (2.2%)

── Spot-check dòng quá ngắn (< 10 ký tự) ──────────
   Tổng dòng          : 1,985
   Dòng quá ngắn     : 23 (1.2%)
   Mẫu: ['Phần I ', 'Chương I ', 'Chương II ', 'Đông. ', 'Hoa. ']

── Sample text (500 ký tự đầu) ─────────────────────
Bộ giáo dục và đào tạo 
Giáo trình 
Triết học mác - lênin (Dùng trong các trường đại học, cao đẳng) (Tái bản lần thứ ba có sửa chữa, bổ sung) 
Đồng chủ biên: 
GS, TS. Nguyễn Ngọc Long - GS, TS. Nguyễn Hữu Vui 
Tập thể tác giả: PGS. TS. Vũ Tình 
PGS.TS. Trần Văn Thụy 
GS, TS. Nguyễn Hữu Vui 
GS, TS. Nguyễn Ngọc Long 
TS. Vương Tất Đạt 
TS. Dương Văn Thịnh 
PGS, TS. Đoàn Quang Thọ 
TS. Nguyễn Như Hải 
PGS, TS. Trương Giang Long 
PGS.TS. Đoàn Đức Hiếu 
TS. Phạm Văn Sinh 
Th.S. Vũ Thanh Bình 
CN. Ng


## Bước 3–4: Segmentation theo cấu trúc + Chunking (underthesea)

- **Segmentation**: tách văn bản thành các `section` theo pattern `Chương/Phần/Mục` bằng regex.  
- **Chunking**: dùng `underthesea.sent_tokenize` để tách câu, sau đó áp dụng sliding window `(max_sentences=5, overlap=1)`.  
- **Mục tiêu**: ~620–680 chunks để ra ~1,500 QA sau lọc.

In [4]:
from underthesea import sent_tokenize


# ── Segmentation ──────────────────────────────────────────────────────────────

def segment_by_structure(text: str) -> List[Dict]:
    """
    Tách văn bản theo cấu trúc Chương / Mục.
    Trả về list sections có: chapter, section, text.
    Fallback về 1 section toàn bộ nếu không tìm thấy pattern.
    """
    chapter_pattern = re.compile(
        r"(?:Chương|CHƯƠNG|Phần|PHẦN)\s*(?:[IVXivx]+|\d+)[:\.\s]+[^\n]+",
        re.IGNORECASE,
    )
    chapter_matches = list(chapter_pattern.finditer(text))

    if not chapter_matches:
        logger.warning("Không tìm thấy pattern Chương/Phần → dùng toàn bộ text")
        return [{"chapter": "General", "section": "N/A", "text": text}]

    sections = []
    for i, ch_match in enumerate(chapter_matches):
        ch_start = ch_match.start()
        ch_end   = chapter_matches[i + 1].start() if i + 1 < len(chapter_matches) else len(text)
        ch_title = ch_match.group(0).strip()[:80]
        ch_text  = text[ch_start:ch_end].strip()

        # Tách mục bên trong chương
        sec_pattern = re.compile(
            r"(?:^|\n)(?:[IVXivx]+|[0-9]+)[.\):\s]+[^\n]{5,}",
            re.MULTILINE,
        )
        sec_matches = list(sec_pattern.finditer(ch_text))

        if not sec_matches:
            sections.append({"chapter": ch_title, "section": "Main", "text": ch_text})
        else:
            for j, sm in enumerate(sec_matches):
                s_start = sm.start()
                s_end   = sec_matches[j + 1].start() if j + 1 < len(sec_matches) else len(ch_text)
                sections.append({
                    "chapter": ch_title,
                    "section": sm.group(0).strip()[:60],
                    "text": ch_text[s_start:s_end].strip(),
                })

    return sections


# ── Chunking với underthesea ──────────────────────────────────────────────────

def chunk_with_sentences(
    sections: List[Dict],
    max_sentences: int = MAX_SENTENCES,
    overlap: int = OVERLAP_SENTENCES,
    min_chars: int = MIN_CHUNK_CHARS,
    max_chars: int = MAX_CHUNK_CHARS,
) -> List[Dict]:
    """
    Chunk từng section bằng underthesea.sent_tokenize + sliding window.
    Mỗi chunk giữ đủ metadata: chunk_id, chapter, section, content, char_count, word_count.
    """
    chunks = []
    chunk_id = 0
    step = max(1, max_sentences - overlap)

    for sec in sections:
        sentences = sent_tokenize(sec["text"])
        sentences = [s.strip() for s in sentences if len(s.strip()) > 8]

        if not sentences:
            continue

        for start in range(0, len(sentences), step):
            window  = sentences[start: start + max_sentences]
            content = " ".join(window).strip()

            if len(content) < min_chars:
                continue

            if len(content) > max_chars:
                # Cắt tại dấu câu gần nhất
                truncated = content[:max_chars]
                last_stop = max(truncated.rfind("."), truncated.rfind("!"), truncated.rfind("?"))
                content   = truncated[:last_stop + 1] if last_stop > max_chars * 0.7 else truncated

            chunks.append({
                "chunk_id"  : f"chunk_{chunk_id:05d}",
                "chapter"   : sec["chapter"],
                "section"   : sec["section"],
                "content"   : content,
                "char_count": len(content),
                "word_count": len(content.split()),
            })
            chunk_id += 1

    return chunks


# ── Chạy ─────────────────────────────────────────────────────────────────────
sections = segment_by_structure(clean_text)
chunks   = chunk_with_sentences(sections)

# ── Stats ─────────────────────────────────────────────────────────────────────
char_counts = [c["char_count"] for c in chunks]
word_counts = [c["word_count"] for c in chunks]

print(f"✅ Segmentation : {len(sections)} sections")
print(f"✅ Chunking     : {len(chunks)} chunks")
print()
print(f"── Chunk stats ─────────────────────────────────────")
print(f"   Avg chars/chunk : {sum(char_counts)/len(char_counts):.0f}")
print(f"   Avg words/chunk : {sum(word_counts)/len(word_counts):.0f}")
print(f"   Min chars       : {min(char_counts)}")
print(f"   Max chars       : {max(char_counts)}")
print()

# Phân bổ theo chương
chapter_dist = Counter(c["chapter"][:50] for c in chunks)
print(f"── Phân bổ chunks theo Chương ──────────────────────")
for ch, cnt in sorted(chapter_dist.items(), key=lambda x: -x[1]):
    bar = "█" * (cnt // 5)
    print(f"   {ch[:50]:<52}  {cnt:>3}  {bar}")
print()

# Cảnh báo nếu chunk count lệch xa target
diff = len(chunks) - TARGET_CHUNKS
if abs(diff) > 100:
    print(f"⚠️  Chunk count ({len(chunks)}) lệch {diff:+d} so với target ({TARGET_CHUNKS}).")
    print("   Điều chỉnh MAX_SENTENCES hoặc MIN_CHUNK_CHARS nếu cần.")
else:
    print(f"✅ Chunk count trong ngưỡng target ({TARGET_CHUNKS} ± 100)")
print()

# Sample 2 chunks đầu
print("── Sample chunk [0] ────────────────────────────────")
print(json.dumps(chunks[0], ensure_ascii=False, indent=2))
print()
print("── Sample chunk [1] ────────────────────────────────")
print(json.dumps(chunks[1], ensure_ascii=False, indent=2))

c:\CS431\UniPolis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Segmentation : 331 sections
✅ Chunking     : 696 chunks

── Chunk stats ─────────────────────────────────────
   Avg chars/chunk : 735
   Avg words/chunk : 163
   Min chars       : 252
   Max chars       : 900

── Phân bổ chunks theo Chương ──────────────────────
   Chương II 
Khái lược về lịch sử triết họctrước mác    135  ███████████████████████████
   Chương VII 
Những cặp phạm trù cơ bản của phép biệ     69  █████████████
   Chương V 
Vật chất và ý thức                           55  ███████████
   Chương VIII 
Những quy luật cơ bản của phép biện c     55  ███████████
   Chương III 
Sự ra đời và phát triển của triết học      51  ██████████
   Chương X 
Hình thái kinh tế - xã hội                   49  █████████
   Chương XIII Ý thức xã hội                              45  █████████
   Chương XI 
Giai cấp và dân tộc                         42  ████████
   Chương IX 
Lý luận nhận thức                           41  ████████
   Chương XII 
Nhà nước và cách mạng xã hội               41 

## Bước 5: Sinh QA bằng Gemini

Mỗi chunk sinh **3 câu hỏi** theo 3 intent:
- `factual` — định nghĩa / sự kiện / khái niệm trực tiếp  
- `relational` — quan hệ, so sánh, nguyên nhân–kết quả  
- `applicative` — ý nghĩa, hệ quả, liên hệ thực tiễn  

**Checkpoint**: kết quả được lưu dần vào `qa_checkpoint.jsonl` sau mỗi chunk để tiếp tục nếu bị ngắt.

In [5]:
from google import genai
from google.genai import types

_client = genai.Client(api_key=os.environ.get("GOOGLE_API_KEY"))

# ── Prompt template ───────────────────────────────────────────────────────────
_QA_PROMPT = """\
Bạn là chuyên gia biên soạn câu hỏi học thuật từ giáo trình đại học các môn Lý luận Chính trị (Triết học Mác-Lênin, Lịch sử Đảng, Pháp luật Đại cương...).

Dựa vào đoạn nội dung bên dưới, hãy sinh CHÍNH XÁC {n} câu hỏi với 3 loại intent:
1. factual      — định nghĩa, sự kiện, khái niệm trực tiếp trong đoạn
2. relational   — mối quan hệ, so sánh, nguyên nhân–kết quả
3. applicative  — ý nghĩa, hệ quả, liên hệ thực tiễn

YÊU CẦU BẮT BUỘC:
- Mỗi câu hỏi PHẢI trả lời được hoàn toàn từ đoạn nội dung được cung cấp.
- Không hỏi về thông tin ngoài đoạn văn.
- Tiếng Việt học thuật, câu hỏi rõ ràng, cụ thể.
- evidence_span là đoạn trích ngắn (< 120 ký tự) trong nội dung trực tiếp trả lời câu hỏi.

NỘI DUNG:
{content}

Trả về JSON hợp lệ theo đúng schema sau (không thêm text ngoài JSON):
{{
  "questions": [
    {{
      "query": "câu hỏi đầy đủ bằng tiếng Việt",
      "intent_type": "factual",
      "evidence_span": "đoạn bằng chứng ngắn trong nội dung"
    }},
    {{
      "query": "...",
      "intent_type": "relational",
      "evidence_span": "..."
    }},
    {{
      "query": "...",
      "intent_type": "applicative",
      "evidence_span": "..."
    }}
  ]
}}"""


# ── Hàm sinh QA cho 1 chunk ───────────────────────────────────────────────────

def generate_qa_for_chunk(
    chunk: Dict,
    client: genai.Client,
    n_queries: int = QUERIES_PER_CHUNK,
    max_retries: int = MAX_RETRIES,
) -> List[Dict]:
    """Gọi Gemini để sinh QA, trả về list câu hỏi đã parse."""
    prompt = _QA_PROMPT.format(n=n_queries, content=chunk["content"])

    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    temperature=GEMINI_TEMPERATURE,
                    max_output_tokens=700,
                ),
            )
            data      = json.loads(response.text)
            questions = data.get("questions", [])

            results = []
            for q in questions[:n_queries]:
                query = q.get("query", "").strip()
                if not query:
                    continue
                results.append({
                    "query"        : query,
                    "intent_type"  : q.get("intent_type", "factual"),
                    "evidence_span": q.get("evidence_span", ""),
                    "positive_id"  : chunk["chunk_id"],
                    "positive"     : chunk["content"],
                    "chapter"      : chunk["chapter"],
                    "section"      : chunk["section"],
                })
            return results

        except Exception as e:
            wait = 2 ** attempt
            if attempt < max_retries - 1:
                logger.warning(f"  Lỗi chunk {chunk['chunk_id']} (attempt {attempt+1}): {e} — retry sau {wait}s")
                time.sleep(wait)
            else:
                logger.error(f"  Bỏ qua chunk {chunk['chunk_id']} sau {max_retries} lần: {e}")

    return []


# ── Hàm sinh QA toàn bộ với checkpoint ───────────────────────────────────────

def generate_all_qa(
    chunks: List[Dict],
    checkpoint_path: Path,
    n_queries: int = QUERIES_PER_CHUNK,
) -> List[Dict]:
    """
    Sinh QA cho tất cả chunks.
    Tự động skip chunk đã có trong checkpoint để tiếp tục nếu bị ngắt giữa chừng.
    """
    # Load checkpoint
    done_ids: set = set()
    all_qa: List[Dict] = []

    if checkpoint_path.exists():
        with open(checkpoint_path, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    item = json.loads(line)
                    all_qa.append(item)
                    done_ids.add(item["positive_id"])
                except json.JSONDecodeError:
                    pass
        print(f"  📂 Checkpoint: {len(all_qa)} QA từ {len(done_ids)} chunks đã hoàn thành")

    remaining = [c for c in chunks if c["chunk_id"] not in done_ids]
    print(f"  ⏳ Còn lại: {len(remaining)} chunks cần sinh QA")
    print()

    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)

    with open(checkpoint_path, "a", encoding="utf-8") as ckpt_f:
        for i, chunk in enumerate(remaining):
            qa_list = generate_qa_for_chunk(chunk, _client, n_queries)

            for qa in qa_list:
                all_qa.append(qa)
                ckpt_f.write(json.dumps(qa, ensure_ascii=False) + "\n")

            ckpt_f.flush()
            time.sleep(SLEEP_BETWEEN_CALLS)

            # Log tiến độ mỗi 50 chunks
            if (i + 1) % 50 == 0 or i == len(remaining) - 1:
                print(f"  [{i+1:>4}/{len(remaining)}] QA tích lũy: {len(all_qa):>5} | chunk: {chunk['chunk_id']}")

    return all_qa


# ── Chạy ─────────────────────────────────────────────────────────────────────
print("🚀 Bắt đầu sinh QA...")
print(f"   Model  : {GEMINI_MODEL}")
print(f"   Chunks : {len(chunks)}")
print(f"   Queries/chunk: {QUERIES_PER_CHUNK}")
print(f"   Est. total QA: {len(chunks) * QUERIES_PER_CHUNK}")
print(f"   Checkpoint: {CHECKPOINT_PATH}")
print()

qa_candidates = generate_all_qa(chunks, CHECKPOINT_PATH)

# ── Output stats ──────────────────────────────────────────────────────────────
print()
print(f"✅ Sinh xong: {len(qa_candidates)} QA candidates")
print()

intent_dist = Counter(q["intent_type"] for q in qa_candidates)
print("── Phân bổ intent ──────────────────────────────────")
for intent, cnt in intent_dist.items():
    pct = cnt / len(qa_candidates) * 100
    print(f"   {intent:<15}: {cnt:>5}  ({pct:.1f}%)")
print()

q_lens = [len(q["query"]) for q in qa_candidates]
print("── Query length stats ──────────────────────────────")
print(f"   Avg : {sum(q_lens)/len(q_lens):.0f} ký tự")
print(f"   Min : {min(q_lens)}")
print(f"   Max : {max(q_lens)}")
print()

# Sample 3 câu hỏi đa dạng intent
print("── Sample QA theo intent ────────────────────────────")
shown = set()
for qa in qa_candidates:
    if qa["intent_type"] not in shown:
        shown.add(qa["intent_type"])
        print(f"\n[{qa['intent_type'].upper()}]")
        print(f"  Query   : {qa['query']}")
        print(f"  Evidence: {qa['evidence_span'][:100]}")
        print(f"  Chunk   : {qa['positive_id']}")
    if len(shown) == 3:
        break

INFO | AFC is enabled with max remote calls: 10.


🚀 Bắt đầu sinh QA...
   Model  : gemini-2.5-flash-lite
   Chunks : 696
   Queries/chunk: 3
   Est. total QA: 2088
   Checkpoint: ..\data\training_data_th\qa_checkpoint.jsonl

  📂 Checkpoint: 366 QA từ 122 chunks đã hoàn thành
  ⏳ Còn lại: 574 chunks cần sinh QA



INFO | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash-lite:generateContent "HTTP/1.1 200 OK"
INFO | AFC is enabled with max remote calls: 10.
INFO | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash-lite:generateContent "HTTP/1.1 200 OK"
INFO | AFC is enabled with max remote calls: 10.
INFO | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash-lite:generateContent "HTTP/1.1 200 OK"
INFO | AFC is enabled with max remote calls: 10.


KeyboardInterrupt: 

## Bước 6–7: Gắn Positive + Negative Sampling

- **Positive**: map trực tiếp câu hỏi → chunk nguồn đã sinh ra nó.  
- **Negatives (3 mức độ)**:
  - `easy`   — chunk từ **chương khác** (dễ phân biệt)
  - `medium` — chunk **cùng chương, khác mục** (vừa)
  - `hard`   — chunk **BM25 top-k** theo query nhưng không phải positive (khó nhất)

In [ ]:
from rank_bm25 import BM25Okapi

random.seed(RANDOM_SEED)


# ── Helpers ───────────────────────────────────────────────────────────────────

def build_chunk_index(chunks: List[Dict]) -> Dict[str, Dict]:
    return {c["chunk_id"]: c for c in chunks}


def tokenize_vi(text: str) -> List[str]:
    """Tokenize đơn giản cho BM25: lowercase + tách từ."""
    return re.sub(r"[^\w\s]", " ", text.lower()).split()


def build_bm25(chunks: List[Dict]) -> BM25Okapi:
    corpus = [tokenize_vi(c["content"]) for c in chunks]
    return BM25Okapi(corpus)


def sample_negatives(
    qa: Dict,
    chunks: List[Dict],
    chunk_index: Dict[str, Dict],
    bm25: BM25Okapi,
    n: int = NEGATIVES_PER_PAIR,
) -> List[str]:
    """
    Lấy n negatives theo 3 mức độ:
      slot 0 → easy  (khác chương)
      slot 1 → medium (cùng chương, khác mục)
      slot 2 → hard   (BM25 top hit ≠ positive)
    """
    pos_id    = qa["positive_id"]
    pos_chunk = chunk_index.get(pos_id)
    if not pos_chunk:
        return []

    negatives: List[str] = []
    used_ids: set = {pos_id}

    # ── Easy: khác chương ─────────────────────────────────────────────────────
    easy_pool = [c for c in chunks if c["chapter"] != pos_chunk["chapter"]]
    if easy_pool:
        pick = random.choice(easy_pool)
        negatives.append(pick["content"])
        used_ids.add(pick["chunk_id"])

    # ── Medium: cùng chương, khác mục ────────────────────────────────────────
    med_pool = [
        c for c in chunks
        if c["chapter"] == pos_chunk["chapter"]
        and c["section"] != pos_chunk["section"]
        and c["chunk_id"] not in used_ids
    ]
    if med_pool:
        pick = random.choice(med_pool)
        negatives.append(pick["content"])
        used_ids.add(pick["chunk_id"])

    # ── Hard: BM25 top hit ≠ positive ─────────────────────────────────────────
    tokens = tokenize_vi(qa["query"])
    scores = bm25.get_scores(tokens)
    sorted_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)

    for idx in sorted_idx:
        candidate = chunks[idx]
        if candidate["chunk_id"] not in used_ids:
            negatives.append(candidate["content"])
            used_ids.add(candidate["chunk_id"])
            break

    # Padding nếu vẫn thiếu (trường hợp corpus nhỏ)
    while len(negatives) < n:
        pool = [c for c in chunks if c["chunk_id"] not in used_ids]
        if not pool:
            break
        pick = random.choice(pool)
        negatives.append(pick["content"])
        used_ids.add(pick["chunk_id"])

    return negatives[:n]


# ── Chạy ─────────────────────────────────────────────────────────────────────
print("🔨 Xây BM25 index cho hard negative mining...")
chunk_index = build_chunk_index(chunks)
bm25        = build_bm25(chunks)
print(f"✅ BM25 index: {len(chunks)} chunks")
print()

print("🔨 Gắn positive + sinh negatives...")
train_pairs_raw: List[Dict] = []
skipped = 0

for qa in qa_candidates:
    if qa["positive_id"] not in chunk_index:
        skipped += 1
        continue

    negs = sample_negatives(qa, chunks, chunk_index, bm25)
    train_pairs_raw.append({
        "query"        : qa["query"],
        "positive"     : qa["positive"],
        "positive_id"  : qa["positive_id"],
        "negatives"    : negs,
        "intent_type"  : qa["intent_type"],
        "evidence_span": qa["evidence_span"],
        "chapter"      : qa["chapter"],
        "section"      : qa["section"],
    })

print(f"✅ Tạo xong {len(train_pairs_raw)} training pairs  (bỏ qua: {skipped})")
print()

# Kiểm tra số lượng negatives
neg_counts = Counter(len(p["negatives"]) for p in train_pairs_raw)
print("── Phân bổ số negatives / pair ─────────────────────")
for cnt, freq in sorted(neg_counts.items()):
    print(f"   {cnt} negatives: {freq} pairs")
print()

# Sample 1 pair đầy đủ
print("── Sample training pair ─────────────────────────────")
sample = train_pairs_raw[0].copy()
sample["negatives"] = [n[:120] + "…" for n in sample["negatives"]]
print(json.dumps(sample, ensure_ascii=False, indent=2))

🔨 Xây BM25 index cho hard negative mining...
✅ BM25 index: 444 chunks

🔨 Gắn positive + sinh negatives...
✅ Tạo xong 1332 training pairs  (bỏ qua: 0)

── Phân bổ số negatives / pair ─────────────────────
   3 negatives: 1332 pairs

── Sample training pair ─────────────────────────────
{
  "query": "Học phần Pháp luật đại cương tại Trường Đại học Thương mại có tính chất như thế nào trong chương trình đào tạo?",
  "positive": "Chương 9\nThư ký nhóm biên soạn\nThS. Phạm Minh Quốc \nChủ biên\nTS. Trần Thành Thọ \nLỜI NÓI ĐẦU\nNhà nước và pháp luật là những hiện tượng xã hội phức tạp và có mối quan hệ gắn bó với nhau. Mỗi phương diện và cách thức thể hiện các yếu tố hợp thành của hai hiện tượng này được những môn khoa học pháp lý khác nhau nghiên cứu, lý giải. Trong quá trình xây dựng nhà nước pháp quyền ở nước ta hiện nay, những nhận thức cơ bản về nhà nước và pháp luật đóng vai trò rất quan trọng đối với việc nâng cao ý thức pháp luật, tạo lập thói quen ứng xử phù hợp với chuẩn mực xã hội

## Bước 8: Quality Filter + Dedup

**Rule-based filter** loại:
- Query quá ngắn / quá dài
- Query mơ hồ (chỉ dẫn về "đoạn trên", "nội dung này"...)
- Pair không có negatives

**Dedup** theo hash normalized query (loại trùng lặp).

In [ ]:
_VAGUE_PATTERNS = [
    re.compile(r"nội dung (trên|này|đó|sau)", re.IGNORECASE),
    re.compile(r"đoạn (văn )?(trên|này|dưới|sau)", re.IGNORECASE),
    re.compile(r"theo (như )?trên", re.IGNORECASE),
    re.compile(r"như (đã )?nêu( ở trên)?", re.IGNORECASE),
    re.compile(r"được đề cập (trên|ở trên)", re.IGNORECASE),
]


def check_query_quality(query: str) -> Tuple[bool, str]:
    """
    Kiểm tra chất lượng query theo rule.
    Trả về (passed: bool, reason: str).
    """
    if len(query) < MIN_QUERY_CHARS:
        return False, "too_short"
    if len(query) > MAX_QUERY_CHARS:
        return False, "too_long"

    for pat in _VAGUE_PATTERNS:
        if pat.search(query):
            return False, "vague"

    # Không có từ nào có nghĩa (chỉ ký tự đặc biệt / số)
    if not re.search(r"[a-zA-ZÀ-ỹ]{3,}", query):
        return False, "no_words"

    return True, "ok"


def dedup_pairs(pairs: List[Dict]) -> List[Dict]:
    """Loại bỏ các pair trùng query (sau normalize)."""
    seen: set = set()
    deduped: List[Dict] = []
    for pair in pairs:
        norm = re.sub(r"\s+", " ", pair["query"].lower().strip())
        key  = hashlib.md5(norm.encode()).hexdigest()
        if key not in seen:
            seen.add(key)
            deduped.append(pair)
    return deduped


# ── Chạy filter ───────────────────────────────────────────────────────────────
filtered: List[Dict] = []
filter_stats: Counter = Counter()

for pair in train_pairs_raw:
    # Pair phải có đủ negatives
    if not pair["negatives"]:
        filter_stats["removed_no_negatives"] += 1
        continue

    passed, reason = check_query_quality(pair["query"])
    if passed:
        filtered.append(pair)
        filter_stats["passed"] += 1
    else:
        filter_stats[f"removed_{reason}"] += 1

print(f"── Filter kết quả ──────────────────────────────────")
for k, v in sorted(filter_stats.items(), key=lambda x: -x[1]):
    pct = v / len(train_pairs_raw) * 100
    print(f"   {k:<30}: {v:>5}  ({pct:.1f}%)")
print()

# ── Dedup ─────────────────────────────────────────────────────────────────────
before_dedup = len(filtered)
deduped = dedup_pairs(filtered)

print(f"── Dedup ───────────────────────────────────────────")
print(f"   Trước dedup : {before_dedup}")
print(f"   Sau dedup   : {len(deduped)}")
print(f"   Đã xóa dup  : {before_dedup - len(deduped)}")
print()
print(f"✅ Dataset cuối: {len(deduped)} pairs")

# Cảnh báo nếu lệch xa target 1,500
target_qa = 1500
diff = len(deduped) - target_qa
if abs(diff) > 200:
    print(f"\n⚠️  Số QA cuối ({len(deduped)}) lệch {diff:+d} so với target ({target_qa}).")
    if diff < 0:
        print("   → Tăng TARGET_CHUNKS hoặc giảm ngưỡng filter.")
    else:
        print("   → Kết quả tốt hơn mong đợi.")
else:
    print(f"✅ Số QA trong ngưỡng target {target_qa} ± 200")
print()

# Intent distribution sau filter
intent_final = Counter(p["intent_type"] for p in deduped)
print("── Intent distribution (sau filter) ───────────────")
for intent, cnt in intent_final.items():
    pct = cnt / len(deduped) * 100
    print(f"   {intent:<15}: {cnt:>5}  ({pct:.1f}%)")

── Filter kết quả ──────────────────────────────────
   passed                        :  1315  (98.7%)
   removed_vague                 :    17  (1.3%)

── Dedup ───────────────────────────────────────────
   Trước dedup : 1315
   Sau dedup   : 1314
   Đã xóa dup  : 1

✅ Dataset cuối: 1314 pairs
✅ Số QA trong ngưỡng target 1500 ± 200

── Intent distribution (sau filter) ───────────────
   factual        :   429  (32.6%)
   applicative    :   443  (33.7%)
   relational     :   442  (33.6%)


## Bước 9–10: Split + Export

**Split theo chương** để tránh data leakage giữa train/val/test.

**Output files:**
| File | Format | Dùng cho |
|------|--------|----------|
| `train_pairs_mnrl.jsonl` | `[query, positive]` | `MultipleNegativesRankingLoss` |
| `train_triplets.jsonl` | `[query, positive, neg1, neg2, neg3]` | TripletLoss / các loss khác |
| `train_data.json` | JSON đầy đủ metadata | Reference / audit |
| `val_data_mnrl.jsonl` | `[query, positive]` | Validation |
| `val_data.json` | JSON đầy đủ | Validation audit |
| `test_data_mnrl.jsonl` | `[query, positive]` | Final eval |
| `test_data.json` | JSON đầy đủ | Final eval audit |
| `qa_audit.jsonl` | JSONL đầy đủ toàn bộ | Review chất lượng |
| `stats.json` | JSON | Thống kê pipeline |

In [ ]:
def split_by_chapter(
    pairs: List[Dict],
    train_ratio: float = TRAIN_RATIO,
    val_ratio: float   = VAL_RATIO,
    seed: int          = RANDOM_SEED,
) -> Tuple[List[Dict], List[Dict], List[Dict]]:
    """
    Chia dataset theo chương để tránh leakage.
    Gom các chương lần lượt vào train/val/test cho đến khi đủ tỷ lệ.
    """
    by_chapter: Dict[str, List[Dict]] = {}
    for pair in pairs:
        ch = pair.get("chapter", "General")
        by_chapter.setdefault(ch, []).append(pair)

    chapters = list(by_chapter.keys())
    rng = random.Random(seed)
    rng.shuffle(chapters)

    total       = len(pairs)
    train_cap   = int(total * train_ratio)
    val_cap     = int(total * val_ratio)

    train, val, test = [], [], []
    for ch in chapters:
        ch_pairs = by_chapter[ch]
        if len(train) < train_cap:
            train.extend(ch_pairs)
        elif len(val) < val_cap:
            val.extend(ch_pairs)
        else:
            test.extend(ch_pairs)

    return train, val, test


# ── Split ─────────────────────────────────────────────────────────────────────
train_set, val_set, test_set = split_by_chapter(deduped)

print(f"── Split (by chapter) ──────────────────────────────")
print(f"   Total  : {len(deduped)}")
print(f"   Train  : {len(train_set):>5}  ({len(train_set)/len(deduped)*100:.1f}%)")
print(f"   Val    : {len(val_set):>5}  ({len(val_set)/len(deduped)*100:.1f}%)")
print(f"   Test   : {len(test_set):>5}  ({len(test_set)/len(deduped)*100:.1f}%)")
print()


# ── Export helpers ────────────────────────────────────────────────────────────

def save_mnrl(pairs: List[Dict], path: Path) -> None:
    """Lưu dạng [query, positive] cho MultipleNegativesRankingLoss."""
    with open(path, "w", encoding="utf-8") as f:
        for p in pairs:
            f.write(json.dumps([p["query"], p["positive"]], ensure_ascii=False) + "\n")


def save_triplets(pairs: List[Dict], path: Path) -> None:
    """Lưu dạng [query, positive, neg1, neg2, ...] cho TripletLoss."""
    with open(path, "w", encoding="utf-8") as f:
        for p in pairs:
            row = [p["query"], p["positive"]] + p["negatives"]
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def save_jsonl(pairs: List[Dict], path: Path) -> None:
    with open(path, "w", encoding="utf-8") as f:
        for p in pairs:
            f.write(json.dumps(p, ensure_ascii=False) + "\n")


def save_json(data, path: Path) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


# ── Export ────────────────────────────────────────────────────────────────────
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Train
save_mnrl(train_set,     OUTPUT_DIR / "train_pairs_mnrl.jsonl")
save_triplets(train_set, OUTPUT_DIR / "train_triplets.jsonl")
save_json(train_set,     OUTPUT_DIR / "train_data.json")

# Val
save_mnrl(val_set,   OUTPUT_DIR / "val_data_mnrl.jsonl")
save_json(val_set,   OUTPUT_DIR / "val_data.json")

# Test
save_mnrl(test_set,  OUTPUT_DIR / "test_data_mnrl.jsonl")
save_json(test_set,  OUTPUT_DIR / "test_data.json")

# Audit (toàn bộ deduped, đầy đủ metadata)
save_jsonl(deduped, OUTPUT_DIR / "qa_audit.jsonl")

# Stats
stats = {
    "total_pairs"       : len(deduped),
    "train"             : len(train_set),
    "val"               : len(val_set),
    "test"              : len(test_set),
    "raw_qa_generated"  : len(qa_candidates),
    "chunks"            : len(chunks),
    "sections"          : len(sections),
    "pages"             : len(pages),
    "queries_per_chunk" : QUERIES_PER_CHUNK,
    "filter_stats"      : dict(filter_stats),
    "intent_distribution": dict(Counter(p["intent_type"] for p in deduped)),
    "chapter_distribution": dict(Counter(p["chapter"][:50] for p in deduped)),
    "config": {
        "gemini_model"      : GEMINI_MODEL,
        "max_sentences"     : MAX_SENTENCES,
        "overlap_sentences" : OVERLAP_SENTENCES,
        "min_chunk_chars"   : MIN_CHUNK_CHARS,
        "max_chunk_chars"   : MAX_CHUNK_CHARS,
        "negatives_per_pair": NEGATIVES_PER_PAIR,
        "train_ratio"       : TRAIN_RATIO,
        "val_ratio"         : VAL_RATIO,
    },
}
save_json(stats, OUTPUT_DIR / "stats.json")


# ── Summary ───────────────────────────────────────────────────────────────────
print(f"── Files đã export → {OUTPUT_DIR.resolve()} ─────────")
files_info = [
    ("train_pairs_mnrl.jsonl", len(train_set), "MultipleNegativesRankingLoss"),
    ("train_triplets.jsonl",   len(train_set), "TripletLoss / hard negatives"),
    ("train_data.json",        len(train_set), "Full metadata"),
    ("val_data_mnrl.jsonl",    len(val_set),   "Validation MNRL"),
    ("val_data.json",          len(val_set),   "Validation full"),
    ("test_data_mnrl.jsonl",   len(test_set),  "Test MNRL"),
    ("test_data.json",         len(test_set),  "Test full"),
    ("qa_audit.jsonl",         len(deduped),   "Toàn bộ — review chất lượng"),
    ("stats.json",             None,           "Pipeline stats"),
]
for fname, cnt, desc in files_info:
    cnt_str = f"{cnt:>5} pairs" if cnt is not None else "      "
    print(f"   {fname:<28}  {cnt_str}   ← {desc}")

print()
print("=" * 60)
print("✅  DATA GENERATION PIPELINE HOÀN THÀNH!")
print("=" * 60)
print(f"   PDF input     : {PDF_PATH}")
print(f"   Pages         : {len(pages)}")
print(f"   Chunks        : {len(chunks)}")
print(f"   QA generated  : {len(qa_candidates)}")
print(f"   QA final      : {len(deduped)}")
print(f"   Train/Val/Test: {len(train_set)} / {len(val_set)} / {len(test_set)}")
print()
print("📌 Bước tiếp theo:")
print("   1. Review qa_audit.jsonl để kiểm tra chất lượng mẫu")
print("   2. Dùng train_pairs_mnrl.jsonl → fine-tune với MultipleNegativesRankingLoss")
print("   3. Dùng val_data_mnrl.jsonl    → evaluation trong quá trình train")

── Split (by chapter) ──────────────────────────────
   Total  : 1314
   Train  :  1051  (80.0%)
   Val    :   263  (20.0%)
   Test   :     0  (0.0%)

── Files đã export → C:\CS431\UniPolis\data\training_data_pldc ─────────
   train_pairs_mnrl.jsonl         1051 pairs   ← MultipleNegativesRankingLoss
   train_triplets.jsonl           1051 pairs   ← TripletLoss / hard negatives
   train_data.json                1051 pairs   ← Full metadata
   val_data_mnrl.jsonl             263 pairs   ← Validation MNRL
   val_data.json                   263 pairs   ← Validation full
   test_data_mnrl.jsonl              0 pairs   ← Test MNRL
   test_data.json                    0 pairs   ← Test full
   qa_audit.jsonl                 1314 pairs   ← Toàn bộ — review chất lượng
   stats.json                             ← Pipeline stats

✅  DATA GENERATION PIPELINE HOÀN THÀNH!
   PDF input     : ../data/phap-luat-dai-cuong-giao-trinh.pdf
   Pages         : 238
   Chunks        : 444
   QA generated  : 1332
